# Chapter 3: ETF Classification Module

This chapter covers ETF classification approaches:
- Rule-based classification
- Machine learning classification
- Multi-dimensional categorization
- Feedback mechanism for optimization

## 1. Setup and Imports

In [ ]:
import datetime
import os
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Add project root to path
project_root = os.path.dirname(os.path.abspath('__file__'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import DATA_DIR, CLASSIFICATION_CONFIG, setup_logging
from utils import DataLoader, ETFClassifier

logger = setup_logging('etf_classification')

## 2. Classification Dimensions

### Market Direction
- Stock (Equity)
- Bond (Fixed Income)
- Commodity
- Currency
- Mixed

### Strategy Type
- Passive Index
- Enhanced Index
- Smart Beta
- Leveraged/Inverse

### Bond ETF Specifics
- Duration: Short, Medium, Long
- Credit Rating: High, Medium, Low

## 3. Rule-Based Classifier

In [ ]:
class RuleBasedClassifier:
    """Rule-based ETF classifier using predefined patterns"""
    
    def __init__(self, rules_file: str = None):
        """Initialize classifier with rules
        
        Args:
            rules_file: Path to JSON rules file
        """
        self.rules = self._load_default_rules()
        if rules_file and Path(rules_file).exists():
            with open(rules_file, 'r', encoding='utf-8') as f:
                custom_rules = json.load(f)
                self.rules.extend(custom_rules.get('rules', []))
    
    def _load_default_rules(self) -> list:
        """Load default classification rules"""
        return [
            # Stock ETF Rules
            {
                'pattern': '沪深300',
                'category': {'market': 'stock', 'index': 'CSI300'}
            },
            {
                'pattern': '上证50',
                'category': {'market': 'stock', 'index': 'SSE50'}
            },
            {
                'pattern': '中证500',
                'category': {'market': 'stock', 'index': 'CSI500'}
            },
            {
                'pattern': '创业板',
                'category': {'market': 'stock', 'index': 'ChiNext'}
            },
            {
                'pattern': '科创板',
                'category': {'market': 'stock', 'index': 'STAR'}
            },
            
            # Bond ETF Rules
            {
                'pattern': '国债',
                'category': {'market': 'bond', 'type': 'treasury'}
            },
            {
                'pattern': '城投',
                'category': {'market': 'bond', 'type': 'municipal'}
            },
            {
                'pattern': '信用债',
                'category': {'market': 'bond', 'type': 'credit'}
            },
            {
                'pattern': '可转债',
                'category': {'market': 'bond', 'type': 'convertible'}
            },
            
            # Commodity ETF Rules
            {
                'pattern': '黄金',
                'category': {'market': 'commodity', 'type': 'gold'}
            },
            
            # QDII Rules
            {
                'pattern': '纳斯达克',
                'category': {'market': 'stock', 'region': 'US', 'index': 'NASDAQ'}
            },
            {
                'pattern': '标普',
                'category': {'market': 'stock', 'region': 'US', 'index': 'SP500'}
            },
            {
                'pattern': '恒生',
                'category': {'market': 'stock', 'region': 'HK', 'index': 'HSI'}
            }
        ]
    
    def classify(self, etf_name: str, etf_info: dict = None) -> dict:
        """Classify ETF based on name patterns
        
        Args:
            etf_name: ETF name
            etf_info: Additional ETF information
            
        Returns:
            Classification result dictionary
        """
        for rule in self.rules:
            if rule['pattern'] in etf_name:
                return rule['category'].copy()
        
        return {'market': 'unknown'}
    
    def classify_batch(self, etf_names: list) -> list:
        """Classify multiple ETFs
        
        Args:
            etf_names: List of ETF names
            
        Returns:
            List of classification results
        """
        return [self.classify(name) for name in etf_names]

# Initialize rule-based classifier
rule_classifier = RuleBasedClassifier()
print("Rule-based classifier initialized")
print(f"Total rules: {len(rule_classifier.rules)}")

## 4. Machine Learning Classifier

In [ ]:
class MLETFClassifier:
    """Machine learning-based ETF classifier"""
    
    def __init__(self):
        """Initialize ML classifier"""
        self.model = None
        self.feedback_data = []
        self._build_model()
    
    def _build_model(self):
        """Build classification pipeline"""
        self.model = Pipeline([
            ('tfidf', TfidfVectorizer(
                max_features=1000,
                ngram_range=(1, 2),
                min_df=2
            )),
            ('clf', MultinomialNB(alpha=0.1))
        ])
    
    def extract_features(self, etf_info: dict) -> str:
        """Extract text features from ETF info
        
        Args:
            etf_info: ETF information dictionary
            
        Returns:
            Combined text features
        """
        features = []
        
        # Add ETF name
        if 'SEC_NAME' in etf_info:
            features.append(str(etf_info['SEC_NAME']))
        
        # Add fund full name
        if 'FUND_FULLNAME' in etf_info:
            features.append(str(etf_info['FUND_FULLNAME']))
        
        # Add benchmark
        if 'FUND_BENCHMARK' in etf_info:
            features.append(str(etf_info['FUND_BENCHMARK']))
        
        # Add investment type
        if 'FUND_INVESTTYPE' in etf_info:
            features.append(str(etf_info['FUND_INVESTTYPE']))
        
        return ' '.join(features)
    
    def train(self, X: list, y: list):
        """Train the classifier
        
        Args:
            X: List of text features
            y: List of labels
        """
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        self.model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = self.model.predict(X_test)
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
    
    def predict(self, etf_info: dict) -> str:
        """Predict ETF category
        
        Args:
            etf_info: ETF information dictionary
            
        Returns:
            Predicted category
        """
        features = self.extract_features(etf_info)
        return self.model.predict([features])[0]
    
    def predict_batch(self, etf_infos: list) -> list:
        """Predict categories for multiple ETFs
        
        Args:
            etf_infos: List of ETF info dictionaries
            
        Returns:
            List of predicted categories
        """
        features = [self.extract_features(info) for info in etf_infos]
        return self.model.predict(features)
    
    def add_feedback(self, etf_info: dict, predicted: str, actual: str):
        """Add feedback for model improvement
        
        Args:
            etf_info: ETF information
            predicted: Predicted category
            actual: Actual category
        """
        self.feedback_data.append({
            'etf_info': etf_info,
            'predicted': predicted,
            'actual': actual,
            'correct': predicted == actual,
            'timestamp': datetime.datetime.now().isoformat()
        })
    
    def save_feedback(self, filepath: str):
        """Save feedback data to file
        
        Args:
            filepath: Output file path
        """
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.feedback_data, f, ensure_ascii=False, indent=2)
    
    def load_feedback(self, filepath: str):
        """Load feedback data from file
        
        Args:
            filepath: Input file path
        """
        if Path(filepath).exists():
            with open(filepath, 'r', encoding='utf-8') as f:
                self.feedback_data = json.load(f)

# Initialize ML classifier
ml_classifier = MLETFClassifier()
print("ML classifier initialized")

## 5. Hybrid Classifier

In [ ]:
class HybridClassifier:
    """Hybrid classifier combining rules and ML"""
    
    def __init__(self, data_loader: DataLoader = None):
        """Initialize hybrid classifier
        
        Args:
            data_loader: DataLoader instance
        """
        self.data_loader = data_loader or DataLoader()
        self.rule_classifier = RuleBasedClassifier()
        self.ml_classifier = MLETFClassifier()
    
    def classify(self, etf_info: dict) -> dict:
        """Classify ETF using hybrid approach
        
        Args:
            etf_info: ETF information dictionary
            
        Returns:
            Classification result
        """
        etf_name = etf_info.get('SEC_NAME', '')
        
        # First try rule-based classification
        rule_result = self.rule_classifier.classify(etf_name, etf_info)
        
        if rule_result.get('market') != 'unknown':
            return {
                'method': 'rule',
                'result': rule_result,
                'confidence': 1.0
            }
        
        # Fall back to ML classification if available
        try:
            ml_result = self.ml_classifier.predict(etf_info)
            return {
                'method': 'ml',
                'result': {'market': ml_result},
                'confidence': 0.8  # Placeholder
            }
        except:
            return {
                'method': 'unknown',
                'result': {'market': 'unknown'},
                'confidence': 0.0
            }
    
    def get_etf_categories(self, dt: str = None) -> pd.DataFrame:
        """Get ETF categories with full classification
        
        Args:
            dt: Date string
            
        Returns:
            DataFrame with ETF categories
        """
        # Get base ETF info
        df = self.data_loader.get_etf_base_info(dt)
        
        if df.empty:
            return df
        
        # Classify each ETF
        classifications = []
        for _, row in df.iterrows():
            result = self.classify(row.to_dict())
            classifications.append({
                'TRADE_CODE': row['TRADE_CODE'],
                'SEC_NAME': row.get('SEC_NAME', ''),
                'classification_method': result['method'],
                'market': result['result'].get('market', 'unknown'),
                'confidence': result['confidence']
            })
        
        return pd.DataFrame(classifications)

# Initialize hybrid classifier
hybrid_classifier = HybridClassifier()
print("Hybrid classifier initialized")

## 6. Bond ETF Duration Classification

In [ ]:
def classify_bond_duration(portfolio_duration: float) -> str:
    """Classify bond ETF by duration
    
    Args:
        portfolio_duration: Portfolio duration in years
        
    Returns:
        Duration category
    """
    if pd.isna(portfolio_duration):
        return 'unknown'
    
    if portfolio_duration < 1:
        return 'ultra_short'
    elif portfolio_duration < 3:
        return 'short'
    elif portfolio_duration < 5:
        return 'medium'
    elif portfolio_duration < 7:
        return 'medium_long'
    else:
        return 'long'

def classify_credit_rating(credit_rating: str) -> str:
    """Classify bond ETF by credit rating
    
    Args:
        credit_rating: Credit rating string
        
    Returns:
        Credit category
    """
    if pd.isna(credit_rating):
        return 'unknown'
    
    credit_rating = str(credit_rating).upper()
    
    if 'AAA' in credit_rating or 'AA' in credit_rating:
        return 'high'
    elif 'A' in credit_rating or 'BBB' in credit_rating:
        return 'medium'
    else:
        return 'low'

# Test duration classification
durations = [0.5, 1.5, 3.5, 6.0, 10.0]
print("Duration Classification Test:")
for d in durations:
    print(f"  {d} years -> {classify_bond_duration(d)}")

## 7. Classification Statistics

In [ ]:
def generate_classification_stats(categories_df: pd.DataFrame) -> dict:
    """Generate classification statistics
    
    Args:
        categories_df: DataFrame with ETF categories
        
    Returns:
        Statistics dictionary
    """
    if categories_df.empty:
        return {}
    
    stats = {
        'total_etfs': len(categories_df),
        'by_market': categories_df['market'].value_counts().to_dict(),
        'by_method': categories_df['classification_method'].value_counts().to_dict(),
        'avg_confidence': categories_df['confidence'].mean()
    }
    
    # Print statistics
    print("Classification Statistics")
    print("=" * 50)
    print(f"Total ETFs: {stats['total_etfs']}")
    print(f"\nBy Market Direction:")
    for market, count in stats['by_market'].items():
        print(f"  {market}: {count}")
    print(f"\nBy Classification Method:")
    for method, count in stats['by_method'].items():
        print(f"  {method}: {count}")
    print(f"\nAverage Confidence: {stats['avg_confidence']:.2f}")
    
    return stats

# Example usage
# categories = hybrid_classifier.get_etf_categories()
# stats = generate_classification_stats(categories)

## 8. Feedback Mechanism

In [ ]:
# Feedback data format
feedback_template = {
    "etf_code": "510300",
    "etf_name": "CSI300 ETF",
    "predicted": {"market": "stock", "index": "CSI300"},
    "actual": {"market": "stock", "index": "CSI300"},
    "correct": True,
    "timestamp": "2024-01-01T00:00:00"
}

print("Feedback Data Template:")
print(json.dumps(feedback_template, indent=2, ensure_ascii=False))

# Save feedback template
feedback_dir = DATA_DIR / 'feedback'
feedback_dir.mkdir(parents=True, exist_ok=True)

template_file = feedback_dir / 'feedback_template.json'
with open(template_file, 'w', encoding='utf-8') as f:
    json.dump(feedback_template, f, indent=2, ensure_ascii=False)
    
print(f"\nFeedback template saved to: {template_file}")

## 9. Summary

This chapter covered:
1. Rule-based classification using pattern matching
2. Machine learning classification with text features
3. Hybrid classification approach
4. Bond ETF specific classifications (duration, credit)
5. Classification statistics and reporting
6. Feedback mechanism for model improvement

### Key Classes
- `RuleBasedClassifier`: Pattern-based classification
- `MLETFClassifier`: Machine learning classification
- `HybridClassifier`: Combined approach

### Best Practices
- Use rules for well-defined patterns
- Apply ML for ambiguous cases
- Collect feedback for continuous improvement
- Validate classifications regularly